In [6]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

IMG_SIZE = (224, 224)
FEATURE_DIM = 1280
LSTM_UNITS = 256
VOCAB_SIZE = 5000

# Pre-trained CNN
cnn = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

cnn.trainable = False

# Frame input
frame_input = layers.Input(
    shape=(224, 224, 3)
)

frame_features = cnn(frame_input)

feature_extractor = Model(
    frame_input,
    frame_features
)

# Sequence of extracted video features
video_input = layers.Input(
    shape=(None, FEATURE_DIM)
)

encoder = layers.LSTM(
    LSTM_UNITS,
    return_state=True
)

_, state_h, state_c = encoder(video_input)

# Caption input
caption_input = layers.Input(
    shape=(None,)
)

embedding = layers.Embedding(
    VOCAB_SIZE,
    256
)(caption_input)

decoder = layers.LSTM(
    LSTM_UNITS,
    return_sequences=True
)

decoder_output = decoder(
    embedding,
    initial_state=[state_h, state_c]
)

output = layers.Dense(
    VOCAB_SIZE,
    activation="softmax"
)(decoder_output)

model = Model(
    [video_input, caption_input],
    output
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │  1,280,000 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 256),     │  1,573,888 │ input_layer_8[0]… │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ (None, None, 256) │    525,312 │ embedding_2[0][0… │
│                     │                   │            │ lstm_4[0][1],     │
│                     │                   │            │ lstm_4[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, None,      │  1,285,000 │ lstm_5[0][0]      │
│                     │ 5000)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,664,200 (17.79 MB)

 Trainable params: 4,664,200 (17.79 MB)

 Non-trainable params: 0 (0.00 B)